necessary libraries

In [67]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [68]:
import timesfm
import os
import re
import math
import pmdarima as pm
from time import strptime
import pandas as pd
import numpy as np
import seaborn as sns
import datetime
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
import inspect
from dateutil.relativedelta import relativedelta
from statsmodels.tsa.holtwinters import ExponentialSmoothing

loading the dataset

In [69]:
# mapping for generational drug names
drug_all_india_map = {
    'AL - Adult' : 'abc-3tc_adult_synthetic.csv',
    'ZL - Adult' : 'azt-3tc_adult_synthetic.csv',
    'ATV - Adult' : 'atv_adult_synthetic.csv',
    'DRV - Adult' : 'drv_adult_synthetic.csv',
    'RTV - Adult' : 'rtv_adult_synthetic.csv',
    'AL - Pediatric' : 'abc-3tc_pediatric_synthetic.csv',
    'ZL - Pediatric' : 'azt-3tc_pediatric_synthetic.csv',
    'LPV 125 Tablets - Pediatric' : 'lpv125_pediatric_synthetic.csv'
}

In [70]:
# loading file name for all drugs and storing in a dictionary for easy access
drug_all_india_files = {drug: os.path.join('..', 'synthetic-data-generation', 'synth_data', filename) for drug, filename in drug_all_india_map.items()}
drug_all_india_files

{'AL - Adult': '../synthetic-data-generation/synth_data/abc-3tc_adult_synthetic.csv',
 'ZL - Adult': '../synthetic-data-generation/synth_data/azt-3tc_adult_synthetic.csv',
 'ATV - Adult': '../synthetic-data-generation/synth_data/atv_adult_synthetic.csv',
 'DRV - Adult': '../synthetic-data-generation/synth_data/drv_adult_synthetic.csv',
 'RTV - Adult': '../synthetic-data-generation/synth_data/rtv_adult_synthetic.csv',
 'AL - Pediatric': '../synthetic-data-generation/synth_data/abc-3tc_pediatric_synthetic.csv',
 'ZL - Pediatric': '../synthetic-data-generation/synth_data/azt-3tc_pediatric_synthetic.csv',
 'LPV 125 Tablets - Pediatric': '../synthetic-data-generation/synth_data/lpv125_pediatric_synthetic.csv'}

defining forecast horizon

In [71]:
forecast_horizon = 18
print(f"Forecast horizon: {forecast_horizon} months")

Forecast horizon: 18 months


prediction models

In [72]:
# Holt-Winters
def holt_winters(training):
    model = ExponentialSmoothing(training, trend='add', seasonal='add', seasonal_periods=12)
    model_fit = model.fit()
    forecast = model_fit.forecast(forecast_horizon)
    return forecast

In [73]:
# TimesFM Model
tfm = timesfm.TimesFm(
      hparams=timesfm.TimesFmHparams(
          backend="mpu",
          per_core_batch_size=32,
          horizon_len=forecast_horizon,
      ),
      checkpoint=timesfm.TimesFmCheckpoint(
          huggingface_repo_id="google/timesfm-1.0-200m-pytorch"),
  )

def timesfm_model(training):
    timesfm_prediction = tfm.forecast([training['doses'].values], freq=[1])
    timesfm_predictions = timesfm_prediction[0][0]
    return timesfm_predictions

def timesfm_ln_model(training, log_transform):
    timesfm_ln_prediction = tfm.forecast([log_transform], freq=[1])
    timesfmlog_predictions = np.exp(timesfm_ln_prediction[0][0]) - 1
    return timesfmlog_predictions

Fetching 3 files: 100%|██████████| 3/3 [00:00<00:00, 68015.74it/s]


In [74]:
# ARIMA
def arima_model(training, log_transform):
    arima_fit = pm.auto_arima(log_transform, start_p=1, start_q=1,
                             max_p=3, max_q=3, m=12,
                             start_P=0, seasonal=True,
                             d=1, D=1, trace=True,
                             error_action='ignore',
                             suppress_warnings=True,
                             stepwise=True)
    arima_predictions_ln = arima_fit.predict(forecast_horizon)
    arima_predictions = np.exp(arima_predictions_ln)
    return arima_predictions

In [ ]:
# ARIMA + ErrorTimesFM
def arima_errortfm_model(training, percentile, log_transform):
    arima_fit = pm.auto_arima(log_transform, start_p=1, start_q=1,
                             max_p=3, max_q=3, m=12,
                             start_P=0, seasonal=True,
                             d=1, D=1, trace=True,
                             error_action='ignore',
                             suppress_warnings=True,
                             stepwise=True)
    arima_predictions_ln = arima_fit.predict(forecast_horizon)
    error = int(percentile/10)

    training_actuals = log_transform[-forecast_horizon:]
    in_sample_predictions = arima_fit.predict_in_sample()[-forecast_horizon:]
    tfm_error_input = training_actuals - in_sample_predictions

    tfm_error_output = tfm.forecast([tfm_error_input], freq=[1])
    quantile_index = int(percentile / 10)

    return np.expm1(
        arima_predictions_ln
        + tfm_error_output[1][0][:, quantile_index]
    )

unified prediction pipeline

In [76]:
def forecast_drug(drug, file):
    df = pd.read_csv(file)
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").set_index("date").asfreq("MS")

    train = df.iloc[:-forecast_horizon]
    test = df.iloc[-forecast_horizon:]

    y_train = train["doses"]
    y_test = test["doses"]

    log_train = np.log1p(y_train.values)
    log_test = np.log1p(y_test.values)

    predictions = pd.DataFrame({
        "date": test.index,
        "actual": y_test.values,
        "Holt-Winters": holt_winters(y_train),
        "TimesFM": timesfm_model(train),
        "TimesFM Log": timesfm_ln_model(train, log_train),
        "ARIMA": arima_model(train, log_train),
        "ARIMA + ErrorTimesFM 30th Percentile": arima_errortfm_model(train, 30, log_train, log_test),
        "ARIMA + ErrorTimesFM 70th Percentile": arima_errortfm_model(train, 70, log_train, log_test),
    })

    predictions.insert(0, "drug", drug)
    return predictions

In [77]:
all_forecasts = pd.concat(
    [forecast_drug(drug, file) for drug, file in drug_all_india_files.items()],
    ignore_index=True
)

/Users/muskaanchugh/epidemics-art-forecasting/.venv/lib/python3.11/site-packages/statsmodels/tsa/holtwinters/model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(


Performing stepwise search to minimize aic
 ARIMA(1,1,1)(0,1,1)[12]             : AIC=inf, Time=0.11 sec
 ARIMA(0,1,0)(0,1,0)[12]             : AIC=-257.266, Time=0.01 sec
 ARIMA(1,1,0)(1,1,0)[12]             : AIC=-305.628, Time=0.05 sec
 ARIMA(0,1,1)(0,1,1)[12]             : AIC=inf, Time=0.07 sec
 ARIMA(1,1,0)(0,1,0)[12]             : AIC=-280.890, Time=0.02 sec
 ARIMA(1,1,0)(2,1,0)[12]             : AIC=-314.282, Time=0.12 sec
 ARIMA(1,1,0)(2,1,1)[12]             : AIC=inf, Time=0.32 sec
 ARIMA(1,1,0)(1,1,1)[12]             : AIC=inf, Time=0.12 sec
 ARIMA(0,1,0)(2,1,0)[12]             : AIC=-289.105, Time=0.07 sec
 ARIMA(2,1,0)(2,1,0)[12]             : AIC=-325.830, Time=0.16 sec
 ARIMA(2,1,0)(1,1,0)[12]             : AIC=-322.035, Time=0.07 sec
 ARIMA(2,1,0)(2,1,1)[12]             : AIC=inf, Time=0.32 sec
 ARIMA(2,1,0)(1,1,1)[12]             : AIC=inf, Time=0.25 sec
 ARIMA(3,1,0)(2,1,0)[12]             : AIC=-329.697, Time=0.14 sec
 ARIMA(3,1,0)(1,1,0)[12]             : AIC=-324.2

/Users/muskaanchugh/epidemics-art-forecasting/.venv/lib/python3.11/site-packages/statsmodels/tsa/holtwinters/model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(


Performing stepwise search to minimize aic
 ARIMA(1,1,1)(0,1,1)[12]             : AIC=inf, Time=0.13 sec
 ARIMA(0,1,0)(0,1,0)[12]             : AIC=-156.390, Time=0.01 sec
 ARIMA(1,1,0)(1,1,0)[12]             : AIC=-197.092, Time=0.05 sec
 ARIMA(0,1,1)(0,1,1)[12]             : AIC=inf, Time=0.08 sec
 ARIMA(1,1,0)(0,1,0)[12]             : AIC=-173.969, Time=0.00 sec
 ARIMA(1,1,0)(2,1,0)[12]             : AIC=-210.598, Time=0.10 sec
 ARIMA(1,1,0)(2,1,1)[12]             : AIC=inf, Time=0.27 sec
 ARIMA(1,1,0)(1,1,1)[12]             : AIC=inf, Time=0.19 sec
 ARIMA(0,1,0)(2,1,0)[12]             : AIC=-188.587, Time=0.04 sec
 ARIMA(2,1,0)(2,1,0)[12]             : AIC=-219.381, Time=0.14 sec
 ARIMA(2,1,0)(1,1,0)[12]             : AIC=-206.805, Time=0.06 sec
 ARIMA(2,1,0)(2,1,1)[12]             : AIC=-223.090, Time=0.29 sec
 ARIMA(2,1,0)(1,1,1)[12]             : AIC=inf, Time=0.19 sec
 ARIMA(2,1,0)(2,1,2)[12]             : AIC=inf, Time=0.50 sec
 ARIMA(2,1,0)(1,1,2)[12]             : AIC=inf, T

/Users/muskaanchugh/epidemics-art-forecasting/.venv/lib/python3.11/site-packages/statsmodels/tsa/holtwinters/model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(


...

Performing stepwise search to minimize aic
 ARIMA(1,1,1)(0,1,1)[12]             : AIC=-271.341, Time=0.14 sec
 ARIMA(0,1,0)(0,1,0)[12]             : AIC=-193.969, Time=0.01 sec
 ARIMA(1,1,0)(1,1,0)[12]             : AIC=-249.356, Time=0.04 sec
 ARIMA(0,1,1)(0,1,1)[12]             : AIC=-272.551, Time=0.05 sec
 ARIMA(0,1,1)(0,1,0)[12]             : AIC=-243.614, Time=0.01 sec
 ARIMA(0,1,1)(1,1,1)[12]             : AIC=-270.827, Time=0.11 sec
 ARIMA(0,1,1)(0,1,2)[12]             : AIC=-270.779, Time=0.28 sec
 ARIMA(0,1,1)(1,1,0)[12]             : AIC=-266.760, Time=0.05 sec
 ARIMA(0,1,1)(1,1,2)[12]             : AIC=-269.140, Time=0.46 sec
 ARIMA(0,1,0)(0,1,1)[12]             : AIC=inf, Time=0.10 sec
 ARIMA(0,1,2)(0,1,1)[12]             : AIC=-271.684, Time=0.12 sec
 ARIMA(1,1,0)(0,1,1)[12]             : AIC=inf, Time=0.10 sec
 ARIMA(1,1,2)(0,1,1)[12]             : AIC=inf, Time=0.22 sec
 ARIMA(0,1,1)(0,1,1)[12] intercept   : AIC=inf, Time=0.13 sec

Best model:  ARIMA(0,1,1)(0,1,1)[12]  

/Users/muskaanchugh/epidemics-art-forecasting/.venv/lib/python3.11/site-packages/statsmodels/tsa/holtwinters/model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(


Performing stepwise search to minimize aic
 ARIMA(1,1,1)(0,1,1)[12]             : AIC=inf, Time=0.15 sec
 ARIMA(0,1,0)(0,1,0)[12]             : AIC=-115.311, Time=0.01 sec
 ARIMA(1,1,0)(1,1,0)[12]             : AIC=-162.762, Time=0.04 sec
 ARIMA(0,1,1)(0,1,1)[12]             : AIC=inf, Time=0.10 sec
 ARIMA(1,1,0)(0,1,0)[12]             : AIC=-139.113, Time=0.01 sec
 ARIMA(1,1,0)(2,1,0)[12]             : AIC=-169.284, Time=0.11 sec
 ARIMA(1,1,0)(2,1,1)[12]             : AIC=inf, Time=0.31 sec
 ARIMA(1,1,0)(1,1,1)[12]             : AIC=inf, Time=0.13 sec
 ARIMA(0,1,0)(2,1,0)[12]             : AIC=-144.079, Time=0.08 sec
 ARIMA(2,1,0)(2,1,0)[12]             : AIC=-171.780, Time=0.12 sec
 ARIMA(2,1,0)(1,1,0)[12]             : AIC=-165.740, Time=0.06 sec
 ARIMA(2,1,0)(2,1,1)[12]             : AIC=inf, Time=0.46 sec
 ARIMA(2,1,0)(1,1,1)[12]             : AIC=inf, Time=0.24 sec
 ARIMA(3,1,0)(2,1,0)[12]             : AIC=-170.451, Time=0.14 sec
 ARIMA(2,1,1)(2,1,0)[12]             : AIC=-175.2

/Users/muskaanchugh/epidemics-art-forecasting/.venv/lib/python3.11/site-packages/statsmodels/tsa/holtwinters/model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(


Performing stepwise search to minimize aic
 ARIMA(1,1,1)(0,1,1)[12]             : AIC=-45.964, Time=0.07 sec
 ARIMA(0,1,0)(0,1,0)[12]             : AIC=21.447, Time=0.00 sec
 ARIMA(1,1,0)(1,1,0)[12]             : AIC=-34.850, Time=0.03 sec
 ARIMA(0,1,1)(0,1,1)[12]             : AIC=-47.689, Time=0.04 sec
 ARIMA(0,1,1)(0,1,0)[12]             : AIC=-19.841, Time=0.01 sec
 ARIMA(0,1,1)(1,1,1)[12]             : AIC=-48.006, Time=0.09 sec
 ARIMA(0,1,1)(1,1,0)[12]             : AIC=-46.245, Time=0.06 sec
 ARIMA(0,1,1)(2,1,1)[12]             : AIC=-46.117, Time=0.18 sec
 ARIMA(0,1,1)(1,1,2)[12]             : AIC=-46.315, Time=0.33 sec
 ARIMA(0,1,1)(0,1,2)[12]             : AIC=-48.284, Time=0.13 sec
 ARIMA(0,1,0)(0,1,2)[12]             : AIC=-20.386, Time=0.13 sec
 ARIMA(1,1,1)(0,1,2)[12]             : AIC=-46.483, Time=0.20 sec
 ARIMA(0,1,2)(0,1,2)[12]             : AIC=-46.534, Time=0.15 sec
 ARIMA(1,1,0)(0,1,2)[12]             : AIC=-39.199, Time=0.10 sec
 ARIMA(1,1,2)(0,1,2)[12]          

In [78]:
# see results
all_forecasts

,drug,date,actual,Holt-Winters,TimesFM,TimesFM Log,ARIMA,ARIMA + ErrorTimesFM 30th Percentile,ARIMA + ErrorTimesFM 70th Percentile
0,AL - Adult,2023-07-01,725252,719487.988749,726322.750000,731535.250000,735888.589670,713972.605075,729092.086847
1,AL - Adult,2023-08-01,720227,738928.463203,727441.937500,735002.375000,736727.634486,716989.717810,733441.762391
2,AL - Adult,2023-09-01,727236,734981.092959,728249.187500,735967.500000,741850.253393,719837.452568,736652.273297
3,AL - Adult,2023-10-01,733418,733006.189498,733827.812500,741374.875000,742849.986153,719805.372267,736678.034741
4,AL - Adult,2023-11-01,749625,740068.067358,732022.250000,737786.875000,756557.488292,732631.102611,749955.935133
...,...,...,...,...,...,...,...,...,...
139,LPV 125 Tablets - Pediatric,2024-08-01,219256,106228.112503,247071.890625,267703.406250,211277.871509,178784.499354,211924.249994
140,LPV 125 Tablets - Pediatric,2024-09-01,215625,82707.372058,242988.703125,266001.125000,203706.721581,172193.651631,204450.255230
141,LPV 125 Tablets - Pediatric,2024-10-01,203378,64583.330389,240106.437500,267443.906250,201549.153143,169914.777995,202057.268561
142,LPV 125 Tablets - Pediatric,2024-11-01,228342,64420.428526,230209.875000,260588.890625,201650.424937,170642.951990,202599.988797


In [79]:
# removing timestamp from date
all_forecasts["date"] = all_forecasts["date"].dt.strftime('%Y-%m')
all_forecasts

,drug,date,actual,Holt-Winters,TimesFM,TimesFM Log,ARIMA,ARIMA + ErrorTimesFM 30th Percentile,ARIMA + ErrorTimesFM 70th Percentile
0,AL - Adult,2023-07,725252,719487.988749,726322.750000,731535.250000,735888.589670,713972.605075,729092.086847
1,AL - Adult,2023-08,720227,738928.463203,727441.937500,735002.375000,736727.634486,716989.717810,733441.762391
2,AL - Adult,2023-09,727236,734981.092959,728249.187500,735967.500000,741850.253393,719837.452568,736652.273297
3,AL - Adult,2023-10,733418,733006.189498,733827.812500,741374.875000,742849.986153,719805.372267,736678.034741
4,AL - Adult,2023-11,749625,740068.067358,732022.250000,737786.875000,756557.488292,732631.102611,749955.935133
...,...,...,...,...,...,...,...,...,...
139,LPV 125 Tablets - Pediatric,2024-08,219256,106228.112503,247071.890625,267703.406250,211277.871509,178784.499354,211924.249994
140,LPV 125 Tablets - Pediatric,2024-09,215625,82707.372058,242988.703125,266001.125000,203706.721581,172193.651631,204450.255230
141,LPV 125 Tablets - Pediatric,2024-10,203378,64583.330389,240106.437500,267443.906250,201549.153143,169914.777995,202057.268561
142,LPV 125 Tablets - Pediatric,2024-11,228342,64420.428526,230209.875000,260588.890625,201650.424937,170642.951990,202599.988797


In [80]:
# round off all columns to no decimal places for better readability
for col in all_forecasts.columns[2:]:
    all_forecasts[col] = all_forecasts[col].round(0).astype(int)

# rename column 'actual' to 'Actual'
all_forecasts.rename(columns={"actual": "Actual"}, inplace=True)
all_forecasts

,drug,date,Actual,Holt-Winters,TimesFM,TimesFM Log,ARIMA,ARIMA + ErrorTimesFM 30th Percentile,ARIMA + ErrorTimesFM 70th Percentile
0,AL - Adult,2023-07,725252,719488,726323,731535,735889,713973,729092
1,AL - Adult,2023-08,720227,738928,727442,735002,736728,716990,733442
2,AL - Adult,2023-09,727236,734981,728249,735968,741850,719837,736652
3,AL - Adult,2023-10,733418,733006,733828,741375,742850,719805,736678
4,AL - Adult,2023-11,749625,740068,732022,737787,756557,732631,749956
...,...,...,...,...,...,...,...,...,...
139,LPV 125 Tablets - Pediatric,2024-08,219256,106228,247072,267703,211278,178784,211924
140,LPV 125 Tablets - Pediatric,2024-09,215625,82707,242989,266001,203707,172194,204450
141,LPV 125 Tablets - Pediatric,2024-10,203378,64583,240106,267444,201549,169915,202057
142,LPV 125 Tablets - Pediatric,2024-11,228342,64420,230210,260589,201650,170643,202600


statistical analysis

In [81]:
totals = (
    all_forecasts
    .drop(columns="date")
    .groupby("drug", as_index=False)
    .sum(numeric_only=True)
)

totals

,drug,Actual,Holt-Winters,TimesFM,TimesFM Log,ARIMA,ARIMA + ErrorTimesFM 30th Percentile,ARIMA + ErrorTimesFM 70th Percentile
0,AL - Adult,13151746,13319769,13212649,13244880,13696555,13290019,13624041
1,AL - Pediatric,60813267,56635659,58106304,58445302,60443844,56707037,60465842
2,ATV - Adult,35693474,39679897,39916384,37908556,41106293,38333569,40936942
3,DRV - Adult,6169875,5810387,6559164,5772073,6285597,5920759,6338704
4,LPV 125 Tablets - Pediatric,4100381,2801511,4604287,4906962,4299149,3673653,4325980
5,RTV - Adult,4400441,3738835,4224995,4072574,3718216,3664010,3844104
6,ZL - Adult,56978976,51321673,50185072,48996628,52627899,51224313,54533908
7,ZL - Pediatric,11805025,10663145,10551764,10964025,10816130,9650281,11044138


In [82]:
percentage_change = totals.copy()

model_columns = percentage_change.columns.difference(["drug", "Actual"])

percentage_change[model_columns] = (
    percentage_change[model_columns]
    .sub(percentage_change["Actual"], axis=0)
    .div(percentage_change["Actual"], axis=0)
    .mul(100)
)

# rounding off the values to 2 decimal places for better readability
percentage_change[model_columns] = percentage_change[model_columns].round(2)

percentage_change = percentage_change.drop(columns="Actual")

percentage_change

,drug,Holt-Winters,TimesFM,TimesFM Log,ARIMA,ARIMA + ErrorTimesFM 30th Percentile,ARIMA + ErrorTimesFM 70th Percentile
0,AL - Adult,1.28,0.46,0.71,4.14,1.05,3.59
1,AL - Pediatric,-6.87,-4.45,-3.89,-0.61,-6.75,-0.57
2,ATV - Adult,11.17,11.83,6.21,15.16,7.40,14.69
3,DRV - Adult,-5.83,6.31,-6.45,1.88,-4.04,2.74
4,LPV 125 Tablets - Pediatric,-31.68,12.29,19.67,4.85,-10.41,5.50
5,RTV - Adult,-15.03,-3.99,-7.45,-15.50,-16.74,-12.64
6,ZL - Adult,-9.93,-11.92,-14.01,-7.64,-10.10,-4.29
7,ZL - Pediatric,-9.67,-10.62,-7.12,-8.38,-18.25,-6.45


Getting Best Models

In [84]:
def print_best_models(percentage_change):
    model_columns = percentage_change.columns.drop("drug")

    for _, row in percentage_change.iterrows():
        errors = row[model_columns].dropna()
        positive_errors = errors[errors >= 0]

        if not positive_errors.empty:
            best_model = positive_errors.idxmin()
        else:
            # Closest negative value to zero.
            best_model = errors.idxmax()

        error = errors[best_model]

        print(
            f"Best model for {row['drug']} is {best_model} "
            f"with error {error:.2f}%"
        )

print_best_models(percentage_change)

Best model for AL - Adult is TimesFM with error 0.46%
Best model for AL - Pediatric is ARIMA + ErrorTimesFM 70th Percentile with error -0.57%
Best model for ATV - Adult is TimesFM Log with error 6.21%
Best model for DRV - Adult is ARIMA with error 1.88%
Best model for LPV 125 Tablets - Pediatric is ARIMA with error 4.85%
Best model for RTV - Adult is TimesFM with error -3.99%
Best model for ZL - Adult is ARIMA + ErrorTimesFM 70th Percentile with error -4.29%
Best model for ZL - Pediatric is ARIMA + ErrorTimesFM 70th Percentile with error -6.45%
